# Chapter 24 Walkthrough: Tree-Based Models, Ensembles & Interpretability

**Book Reference:** Chapter 24, Sections 24.1 – 24.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every workflow uses `mch_survey_sample` plus Scikit-Learn's built-in tree, ensemble, and permutation-importance APIs — plus exact 3-feature Shapley attribution from scratch.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q sharukh-tutor
import itertools
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import (
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
import sharukh_tutor as tutor

mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = mch["residence"].astype(str).str.strip().str.title()
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3, "Richer": 4, "Richest": 5
}
mch["wealth_rank"] = mch["wealth_quintile"].map(wealth_map)

features = [
    "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
    "is_urban", "wealth_rank",
]
clean = mch[features + ["child_stunted"]].dropna()
X = clean[features]
y = clean["child_stunted"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("Clean complete-case shape:", clean.shape)
print(f"Train: {len(X_train)}, Test: {len(X_test)}, "
      f"Stunting rate: {y.mean():.4f}")


## Section 24.3.1 — Gini Impurity, Shannon Entropy & Greedy Stump Split

In [ ]:
def gini_demo(y_vals):
    arr = np.asarray(y_vals)
    if arr.size == 0:
        return 0.0
    _, counts = np.unique(arr, return_counts=True)
    probs = counts / arr.size
    return round(float(1.0 - np.sum(probs ** 2)), 4)

def entropy_demo(y_vals):
    arr = np.asarray(y_vals)
    if arr.size == 0:
        return 0.0
    _, counts = np.unique(arr, return_counts=True)
    probs = counts / arr.size
    return round(float(-np.sum(probs * np.log2(probs))), 4)

def stump_demo(x_vals, y_vals):
    x = np.asarray(x_vals, dtype=float)
    yv = np.asarray(y_vals)
    uniq = np.unique(x)
    mids = (uniq[:-1] + uniq[1:]) / 2.0
    best_t, best_g = float(mids[0]), float("inf")
    for t in mids:
        left, right = yv[x <= t], yv[x > t]
        wg = (len(left) / len(yv)) * gini_demo(left)
        wg += (len(right) / len(yv)) * gini_demo(right)
        if wg < best_g:
            best_g, best_t = wg, float(t)
    return (round(best_t, 4), round(best_g, 4))

print("Root Gini:", gini_demo(y_train),
      "| Root Entropy:", entropy_demo(y_train))
for col in ["wealth_rank", "anc_visits", "maternal_hb_g_dl"]:
    t, g = stump_demo(X_train[col], y_train)
    print(f"Best stump on {col:16s}: thresh={t:.2f}, w_gini={g:.4f}")


## Section 24.3.2 — Decision Tree Overfitting & Depth Sweep

In [ ]:
depth_rows = []
for d in [1, 2, 3, 5, 8, 12]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    tr_auc = roc_auc_score(y_train, dt.predict_proba(X_train)[:, 1])
    te_auc = roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1])
    depth_rows.append({
        "max_depth": d,
        "train_auc": round(float(tr_auc), 4),
        "val_auc": round(float(te_auc), 4),
        "gap": round(float(tr_auc - te_auc), 4),
    })
print(pd.DataFrame(depth_rows).to_string(index=False))


## Section 24.3.3 — Bagging, Random Forests & Out-of-Bag (OOB) Validation

In [ ]:
oob_rows = []
for d in [2, 4, 6, 8, 12]:
    rf = RandomForestClassifier(
        n_estimators=120, max_depth=d, oob_score=True, random_state=42
    ).fit(X_train, y_train)
    oob_rows.append({
        "max_depth": d,
        "oob_score": round(float(rf.oob_score_), 4),
    })
print(pd.DataFrame(oob_rows).to_string(index=False))


## Section 24.3.4 — Gradient Boosting Residuals, Native NaNs & Ensemble Scaling

In [ ]:
# 1. Sequential residual fitting intuition (3 boosting rounds)
y_hb = np.array([9.4, 10.1, 11.2, 12.6, 13.0])
x_anc = np.array([[0], [1], [3], [6], [8]])
pred = np.full_like(y_hb, y_hb.mean())
lr = 0.5
print(f"Step 0 (mean)   MSE: {np.mean((y_hb - pred) ** 2):.4f}")
for step in range(1, 4):
    resid = y_hb - pred
    stump = DecisionTreeRegressor(max_depth=1, random_state=42)
    stump.fit(x_anc, resid)
    pred = pred + lr * stump.predict(x_anc)
    print(f"Step {step} (boost) MSE: {np.mean((y_hb - pred) ** 2):.4f}")

# 2. Native NaN handling of HistGradientBoosting on all 500 MCH rows
X_full = mch[features]
y_full = mch["child_stunted"]
Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_full, y_full, test_size=0.25, stratify=y_full, random_state=42
)
hgb_nan = HistGradientBoostingClassifier(max_depth=4, random_state=42)
hgb_nan.fit(Xf_tr, yf_tr)
nan_auc = roc_auc_score(yf_te, hgb_nan.predict_proba(Xf_te)[:, 1])
n_nans = int(X_full.isna().sum().sum())
print(f"HGB native NaN (n={len(X_full)}, NaNs={n_nans}) "
      f"test ROC-AUC: {nan_auc:.4f}")

# 3. Ensemble scaling on a 600-row nonlinear cohort
X_syn, y_syn = make_classification(
    n_samples=600, n_features=6, n_informative=4, n_redundant=1,
    weights=[0.80, 0.20], random_state=42,
)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
    X_syn, y_syn, test_size=0.25, stratify=y_syn, random_state=42
)
specs = [
    ("DecisionTree", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("RandomForest", RandomForestClassifier(
        n_estimators=100, max_depth=6, random_state=42)),
    ("HistGradientBoosting", HistGradientBoostingClassifier(
        max_depth=4, random_state=42)),
]
for name, clf in specs:
    clf.fit(Xs_tr, ys_tr)
    p = clf.predict_proba(Xs_te)[:, 1]
    print(f"Synth {name:20s} ROC={roc_auc_score(ys_te, p):.4f} "
          f"PR={average_precision_score(ys_te, p):.4f}")


## Section 24.3.5 — Interpretability: Permutation Importance vs. Default MDI Bias

In [ ]:
rng = np.random.default_rng(42)
noisy = clean.copy()
noisy["random_noise_id"] = rng.uniform(0, 1000, size=len(noisy))
f_noise = features + ["random_noise_id"]
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(
    noisy[f_noise], noisy["child_stunted"],
    test_size=0.25, stratify=noisy["child_stunted"], random_state=42,
)
rf_noise = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42
).fit(Xn_tr, yn_tr)
pi_noise = permutation_importance(
    rf_noise, Xn_te, yn_te, scoring="roc_auc",
    n_repeats=10, random_state=42,
)
imp_df = pd.DataFrame({
    "feature": f_noise,
    "mdi_impurity": np.round(rf_noise.feature_importances_, 4),
    "perm_auc_mean": np.round(pi_noise.importances_mean, 4),
    "perm_auc_std": np.round(pi_noise.importances_std, 4),
}).sort_values("mdi_impurity", ascending=False)
print(imp_df.to_string(index=False))


## Section 24.3.6 — Exact 3-Feature Shapley Values (SHAP Intuition) From Scratch

In [ ]:
top3 = ["wealth_rank", "anc_visits", "maternal_hb_g_dl"]
rf3 = RandomForestClassifier(
    n_estimators=100, max_depth=4, random_state=42
).fit(X_train[top3].to_numpy(), y_train.to_numpy())

def rf3_proba(v):
    arr = np.asarray(v, dtype=float).reshape(1, -1)
    return float(rf3.predict_proba(arr)[0, 1])

def shapley_3_demo(predict_fn, x_row, base_row):
    x = np.asarray(x_row, dtype=float)
    base = np.asarray(base_row, dtype=float)
    base_val = float(predict_fn(base))
    pred_val = float(predict_fn(x))
    contribs = np.zeros(3, dtype=float)
    perms = list(itertools.permutations([0, 1, 2]))
    for perm in perms:
        state = base.copy()
        prev = base_val
        for idx in perm:
            state[idx] = x[idx]
            curr = float(predict_fn(state))
            contribs[idx] += curr - prev
            prev = curr
    shap_vals = contribs / len(perms)
    return {
        "base_value": round(base_val, 4),
        "shap_values": [round(float(v), 4) for v in shap_vals],
        "prediction": round(pred_val, 4),
    }

baseline = X_train[top3].median().to_numpy()
high_risk = np.array([1.0, 1.0, 9.5])
sh = shapley_3_demo(rf3_proba, high_risk, baseline)
print("Baseline (median):", np.round(baseline, 2).tolist())
print("High-risk patient:", high_risk.tolist())
print("Base:", sh["base_value"], "| Pred:", sh["prediction"])
print("Shapley values:", sh["shap_values"])
eff = round(sh["base_value"] + sum(sh["shap_values"]), 4)
print(f"Efficiency check: {eff} ~= {sh['prediction']}")


## Section 24.3.7 — Break It On Purpose! (Tree & Ensemble Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError/InvalidParameterError — max_depth=0
bad_tree = DecisionTreeClassifier(max_depth=0, random_state=42)
bad_tree.fit(X_train, y_train)


In [ ]:
# Intentional Bug #2: ValueError — oob_score=True with bootstrap=False
bad_rf = RandomForestClassifier(
    n_estimators=50, oob_score=True, bootstrap=False, random_state=42
)
bad_rf.fit(X_train, y_train)


In [ ]:
# Intentional Bug #3: ValueError — NaNs passed to classic GBC
bad_gbc = GradientBoostingClassifier(random_state=42)
bad_gbc.fit(Xf_tr, yf_tr)


## Section 24.4 — Stage 4 Blueprint: End-to-End Stunting Tree Audit Pipeline

In [ ]:
def run_stunting_tree_audit(raw_df, seed=42):
    """Clean MCH data, benchmark 3 tree models, and audit top driver."""
    work = raw_df.copy()
    work["residence"] = (
        work["residence"].astype(str).str.strip().str.title()
    )
    work["anc_visits"] = work["anc_visits"].replace(999, np.nan)
    work["is_urban"] = (work["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    work["wealth_rank"] = (
        work["wealth_quintile"].astype(str).str.strip().str.title()
        .map(w_map)
    )
    feats = [
        "maternal_age_years", "anc_visits", "maternal_hb_g_dl",
        "is_urban", "wealth_rank",
    ]
    df = work[feats + ["child_stunted"]].dropna()
    X_tr, X_te, y_tr, y_te = train_test_split(
        df[feats], df["child_stunted"].astype(int),
        test_size=0.25, stratify=df["child_stunted"], random_state=seed,
    )
    models = [
        ("DecisionTree", DecisionTreeClassifier(
            max_depth=5, random_state=seed)),
        ("RandomForest", RandomForestClassifier(
            n_estimators=100, max_depth=6, random_state=seed)),
        ("HistGradientBoosting", HistGradientBoostingClassifier(
            max_depth=4, random_state=seed)),
    ]
    rows, fitted = [], {}
    for name, clf in models:
        clf.fit(X_tr, y_tr)
        proba = clf.predict_proba(X_te)[:, 1]
        rows.append({
            "model": name,
            "roc_auc": round(float(roc_auc_score(y_te, proba)), 4),
            "pr_auc": round(
                float(average_precision_score(y_te, proba)), 4
            ),
        })
        fitted[name] = clf
    board = (
        pd.DataFrame(rows)
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )
    winner = str(board.iloc[0]["model"])
    pi = permutation_importance(
        fitted[winner], X_te, y_te, scoring="roc_auc",
        n_repeats=10, random_state=seed,
    )
    top_feat = feats[int(np.argmax(pi.importances_mean))]
    return board, winner, top_feat

board, winner, top_feat = run_stunting_tree_audit(mch_raw, seed=42)
print(board.to_string(index=False))
print(f"Winning model: {winner} | Top held-out driver: {top_feat}")
